# Build and inspect a Visual Policy Map

**Evidence state:** Defined

## What this demonstrates

Compile the bounded arcade policy into one deterministic VPM, render its normalized field, and trace a selected action back to its source row and metric.

## Why it matters

The result is not only an action. It carries an addressable proof: artifact identity, source row, source metric, view coordinate, raw value, and candidates.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "VERSION").is_file():
    if ROOT.parent == ROOT:
        raise RuntimeError("ZeroModel repository root not found")
    ROOT = ROOT.parent
os.chdir(ROOT)
for path in (ROOT, ROOT / "examples"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
print(f"repository root: {ROOT}")


## Source and package mapping

- `examples/arcade_shooter_policy.py`
- `zeromodel`
- `zeromodel-video`


In [ ]:
from importlib.metadata import version
import json
from IPython.display import Image, display
from examples.arcade_shooter_policy import ACTIONS, ShooterConfig, compile_policy_artifact, run_policy_episode
from zeromodel.core import png_bytes
from zeromodel.core.policy_lookup import VPMPolicyLookup

config = ShooterConfig()
artifact = compile_policy_artifact(config)
reader = VPMPolicyLookup(artifact, action_metric_ids=ACTIONS)
print(json.dumps({"zeromodel": version("zeromodel"), "artifact_id": artifact.artifact_id, "rows": len(artifact.source.row_ids), "metrics": list(artifact.source.metric_ids)}, indent=2))
display(Image(data=png_bytes(artifact), width=220))


In [ ]:
row_id = artifact.source.row_ids[0]
decision = reader.read(row_id)
print(json.dumps({"row_id": row_id, "decision": decision.to_dict()}, indent=2, sort_keys=True))


## Application

```text
scored states -> declared VPM layout -> addressable lookup -> action plus proof
```


In [ ]:
episode = run_policy_episode(config)
print(json.dumps({"score": episode["score"], "cleared": episode["cleared"], "steps": episode["steps"], "first_decisions": episode["trace"][:3]}, indent=2))


## Boundaries and limitations

This demonstrates deterministic compilation, lookup, rendering, and mapping inside a bounded fixture. It does not prove policy optimality or open-world robustness.

## Reproduction record

The builder records the source notebook, command, ZeroModel version, Git revision, executed notebook, and HTML under `docs/results/demos/vpm-artifact/`.


In [ ]:
print(json.dumps({"demo_id": "vpm-artifact", "source": "demos/notebooks/01-vpm-artifact.ipynb", "artifact_id": artifact.artifact_id}, indent=2))
